
**+++++++++++++++++++++++++++++++++++++++++++++++++++++++**

<strong> **Day 4: File I/O, JSON, CSV, and context managers**</strong>

**+++++++++++++++++++++++++++++++++++++++++++++++++++++++**

<strong> **Concept: JSON — Your Agent's Data Exchange Format**</strong>


**<b>JSON (JavaScript Object Notation) </b> **

<h6> is the universal language for data exchange. 
Every API you'll call — OpenAI, Anthropic, FastAPI services, LangChain tools — speaks JSON. 

If you can read and write JSON fluently, you can talk to anything. 

<b><u>Part 1: What JSON Looks Like </u></b>

JSON is a text format for data. It looks almost like Python dictionaries and lists, but with stricter rules.

In [ ]:
# json

{
  "name": "sales_q4",
  "rows": 250000,
  "columns": 18,
  "clean": True,
  "null_rates": {"revenue": 0.02, "country": 0.0}
}

<h6>
<b> Rules: </b>

- Keys must be strings in double quotes.

- Strings must use double quotes (not single).

- Values can be: string, number, boolean (true/false), null, object ({}), or array ([]).

- No trailing commas.

- No comments.

That's it. Simple, strict, and universal.

<b><u> Part 2: Python ↔ JSON Mapping</u> </b>

When Python converts between the two, it maps types like this:

| Python       | JSON       |
|--------------|------------|
| `dict`        | object `{}` |
| `list`, `tuple` | array `[]` |
| `str`         | string `""` |
| `int`, `float` | number     |
| `True` / `False` | `true` / `false` |
| `None`        | `null`     |

JSON has no concept of:

- datetime

- Custom class instances

- bytes

- set

- numpy types

* ! These will raise TypeError unless you handle them — which is the second half of the concept.

<b><u> Part 3: The Four Functions</u></b>

The json module gives you four functions. The names differ by one letter:

| Function              | Input / Output                  | Use                     |
|-----------------------|---------------------------------|-------------------------|
| `json.dumps(obj)`     | Python object → string          | When you need a string  |
| `json.loads(s)`       | String → Python object          | When you have a string  |
| `json.dump(obj, f)`   | Python object → file            | When you write to a file|
| `json.load(f)`        | File → Python object             | When you read from a file|

Mnemonic:

- <b>dumps / loads</b> end in s → work with strings.

- <b> dump / load (no s) </b>→ work with file objects.

- One letter, completely different interface.



<b><u> Part 4: Line-by-Line — Serialization </u></b>

In [ ]:
#python
import json

profile_data = {
    "name": "sales_q4",
    "rows": 250000,
    "columns": 18,
    "clean": True,
    "null_rates": {"revenue": 0.02, "country": 0.0}
}
# A regular Python dict. Values are Python types.

json_string = json.dumps(profile_data, indent=2)

# json.dumps → "dump string."
# Converts the dict into a JSON-formatted string.
# indent=2 makes it pretty-printed with 2-space indentation.
# Without indent, it produces a single line: {"name": "sales_q4", ...}.

In [ ]:
# Result

{
  "name": "sales_q4",
  "rows": 250000,
  "columns": 18,
  "clean": true,
  "null_rates": {
    "revenue": 0.02,
    "country": 0.0
  }
}

# Notice: True became true, and the layout is nested.

<b><u> Part 5: Line-by-Line — Deserialization</u><b>

In [ ]:
# python
loaded = json.loads(json_string)
# json.loads → "load string."
# Parses the JSON string back into a Python dict.
# loaded is now a Python dict, exactly like the original.

print(loaded["null_rates"]["revenue"])   # 0.02
# Access works exactly like a Python dict, because it is a Python dict.
# Nested access works the same way.

<b><u> Part 6: Writing and Reading Files</u></b>

In [ ]:
with open("profile.json", "w") as f:
    json.dump(profile_data, f, indent=2)
    
# open(..., "w") — open for writing.
# json.dump(obj, f, indent=2) — write the JSON directly to the file object.
# No s on dump → it writes to a file, not returns a string.

with open("profile.json", "r") as f:
    loaded = json.load(f)
    
# open(..., "r") — open for reading.
# json.load(f) — parse the file object as JSON, return a Python object.
# No s on load → it reads from a file, not a string.
# This pattern — with open(...) as f: json.dump/load(...) — is the standard way to persist Python data as JSON.

<b><u>Part 7: Why You Can't Always Serialize</u></b>

<h6>JSON only knows about the primitive types listed in Part 2. If you try to serialize something else, you get a TypeError.</h6>

In [ ]:
import json
import numpy as np

data = {"mean": np.float64(142.5)}
json.dumps(data)
# TypeError: Object of type float64 is not JSON serializable

<b>Why?</b>

<h6> np.float64 looks like a float but is not Python's built-in float. 

The JSON encoder checks :

- isinstance(obj, (dict, list, str, int, float, bool, NoneType)) and np.float64 fails every check.

You'll hit this constantly in data/ML code. Pandas values, NumPy values, custom class instances, datetimes — none of them serialize by default.</h6>

<b><u> Part 8: The Custom Encoder </u></b>

<h6> The fix is a custom encoder: a subclass of json.JSONEncoder that knows how to handle the types JSON doesn't.</h6>


In [ ]:
# python
class AgentEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.integer):
            return int(obj)
        return super().default(obj)

In [ ]:
# Line by line Explanation:
# python
class AgentEncoder(json.JSONEncoder):
# Subclass Python's built-in JSONEncoder.

# python
    def default(self, obj):
# default is called by json.dumps when it encounters a type it doesn't know.
# It receives the problematic object as obj.

        if isinstance(obj, np.floating):
            return float(obj)
# np.floating is the base class for all NumPy float types (np.float64, np.float32, etc.).
# Converts the NumPy float to a native Python float, which JSON can serialize.

# python
        if isinstance(obj, np.integer):
            return int(obj)
# Same idea, for NumPy ints.

# python
        return super().default(obj)
    
# If the object isn't one we know, delegate to the parent class's default.
# The parent will raise TypeError, which is the default behavior — you don't silence errors you don't understand.

In [ ]:
# Now:
# python
json.dumps(data, cls=AgentEncoder)
# cls=AgentEncoder tells json.dumps to use your encoder instead of the default.
# The encoder pattern is extensible. Whenever you hit a new unsupported type, 
# add an if isinstance(...) branch. Datetimes, sets, custom classes — same pattern.

<b><u>Part 9: The Design Pattern — Extending default</u></b>

<h6>
Why <b><i> super().default(obj) </i></b> instead of just raising an error yourself?

Because the parent class already knows how to raise the correct TypeError with a helpful message like:

* text

<b><i>
TypeError: Object of type Decimal is not JSON serializable </i></b>

If you raise your own error, you'd lose that message. Delegating to the parent keeps the standard behavior for unknown types.

<b>
Rule: handle what you know; delegate everything else.</b>
</h6>

<b><u>Part 10: Common Pitfalls</u></b>
<h6>

| Pitfall                              | What goes wrong |
|--------------------------------------|-----------------|
| Using `dumps` when you meant `dump`  | You get a string instead of writing to the file. |
| Using `load` when you meant `loads`  | You pass a string to a function that expects a file object, causing a `TypeError`. |
| Forgetting `indent=2`                | Output is one long line and hard to read. |
| Not handling NumPy/Pandas types      | A `TypeError` occurs during serialization. |
| Using single quotes in JSON by hand  | JSON requires double quotes. |
| Adding trailing commas               | JSON forbids trailing commas. |
| Storing `datetime` without conversion | A `TypeError` occurs because JSON has no date type. |
</h6>


### ** SUMMARY **
<h6>
JSON is a text format for data exchange. Python's json module gives you four functions: dumps/loads for strings, dump/load for files. dump writes to a file, load reads from a file, both use with open(...). Types JSON doesn't know about — like NumPy floats and ints — require a custom encoder that extends json.JSONEncoder and overrides default. The encoder checks isinstance and converts unknown types to native Python types; unknown of the unknown delegate to the parent class.
</h6>

<strong> Concept: CSV — Reading Without Pandas </strong>

<h6> You've used <b><i> pandas.read_csv()</i></b> before. Now you'll learn Python's built-in csv module — the lighter, faster, dependency-free way to read and write CSV files. When you don't need pandas' DataFrame power (and its overhead), csv is the right tool.

<b><u> Part 1: What CSV Is </u></b>

<b>CSV</b> stands for <b><i>Comma-Separated Values</i></b>.

It's a plain text format where:

- Each line is a row of data.

- Each comma separates fields within a row.

- The first row typically contains column names (the "header").

Example — sales.csv:

<i> text
name,revenue,country

Widget,1500.50,US

Gadget,2300.75,UK

Gizmo,900.25,DE </i>

Three rows, three columns, header on top.

<b><u> Why not always use pandas?</u></b>

Pandas is fantastic — but it's a large dependency (NumPy, hundreds of MB). For scripts that just need to read a small file, importing pandas costs time and memory. The csv module:

- Is in the standard library — no installation.

- Loads in microseconds.

- Has no heavy dependencies.

Use csv when:

- You're in a small script.

- You're inside an agent or microservice where every MB matters.

- You're processing files line by line (streaming).

- You don't need pandas' analysis power.

Use pandas when:

- You need DataFrames, groupby, joins, plotting.

- You're doing real data analysis.

<b><u>Part 2: The newline="" Rule (Windows)</u></b>

This is the single most important thing to remember about the csv module:

<b><i> Always use open(..., newline="")</i></b> when working with CSV.

Why? The csv module handles line endings itself. If you let Python's file reader also translate line endings (which it does by default on Windows), the two systems conflict and you get blank lines between every row on Windows.

The rule: whenever you use<b> csv.DictReader </b>or <b>csv. DictWriter</b>, add <b>newline=""</b> to open(). </h6>

In [ ]:
with open("sales.csv", "r", newline="") as f:    # reading
    ...

with open("quality_report.csv", "w", newline="") as f:    # writing
    ...
    
# On Linux and macOS, this doesn't cause problems. On Windows, 
# it's the difference between working and broken output.

<h6>
<b><u> Part 3: Reading CSV — Two Ways</u></b>

<b>Way 1:<i> csv.reader </i></b>— each row as a list </h6>

In [ ]:
import csv

with open("sales.csv", "r", newline="") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)
        
# Output:
'''
['name', 'revenue', 'country']
['Widget', '1500.50', 'US']
['Gadget', '2300.75', 'UK']
['Gizmo', '900.25', 'DE'] '''

<h6>

- Each row is a list of strings.

- The header row is included — you have to skip it yourself if you don't want it.

- Access by index: row[0] is name, row[1] is revenue.

- All values are strings. You convert them to numbers yourself.

This is low-level. Fine for simple cases, but you'll lose track of which index is which column.

<b><u> Way 2: csv.DictReader — each row as a dict</u></b>

In [ ]:
with open("sales.csv", "r", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row["revenue"], row["country"])
        
# output:
'''
1500.50 US
2300.75 UK
900.25 DE '''

<h6>

- The first row becomes the column names.

- Each subsequent row is a dict mapping column name to value.

- Access by column name — much clearer than index.

- The header row is not included in the iteration — DictReader consumed it automatically.

<b> This is the way you'll use 95% of the time.</b>

<b> Accessing by column name vs. index </b>

Without DictReader:

In [ ]:
for row in csv.reader(f):
    print(row[1], row[2])   # which column is 1? which is 2?

<h6> with DictReader</h6>

In [ ]:
for row in csv.DictReader(f):
    print(row["revenue"], row["country"])   # obvious
    
# The named version is self-documenting. Use it.

<h6>
<b><u> Reading all rows into a list</u></b>

If you want to keep the rows around after the file closes:</h6>

In [ ]:
with open("sales.csv", "r", newline="") as f:
    rows = list(csv.DictReader(f))

# file is closed, but `rows` is a list of dicts in memory
for row in rows:
    print(row["name"], row["revenue"])

<h6>
<i> list()</i> forces the reader to produce all rows at once. Useful when you need to iterate multiple times or access rows by position.

<b><u> Part 4: Values Are Always Strings</u></b>

This catches everyone the first time.</h6>

In [ ]:
with open("sales.csv", "r", newline="") as f:
    for row in csv.DictReader(f):
        print(row["revenue"] + 1)   # TypeError!
        
# CSV has no concept of "number." Every value read is a string. 
# row["revenue"] is "1500.50", not 1500.50.

# You must convert:
#python

revenue = float(row["revenue"])
is_clean = row["is_clean"] == "True"    # string comparison

# For booleans in CSV:
# Written as "True" or "False" (Python style).
# Read back as strings.
# Convert with row["is_clean"] == "True" or a small helper.
# This is a fundamental difference from pandas, which auto-detects types. With csv, you own the conversion.

<h6><b><u>Part 5: Writing CSV — csv.DictWriter</u></b></h6>

In [ ]:
import csv

columns = ["name", "null_rate", "is_clean"]
rows = [
    {"name": "revenue", "null_rate": 0.02, "is_clean": True},
    {"name": "country", "null_rate": 0.00, "is_clean": True},
]

with open("quality_report.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(rows)
    
# The result file:
'''
name,null_rate,is_clean
revenue,0.02,True
country,0.0,True '''

<h6>
Notice:

- 0.00 becomes 0.0 — Python's float repr.

- True is written as the string "True".

- Commas separate fields.

If you want different formatting (e.g., always 2 decimal places), you need to convert values to strings yourself before writing. csv.DictWriter uses str() on everything.

<b><u> Part 6: A Full Round-Trip Example</u></b>

Write, then read, then verify.</h6>

In [ ]:
import csv

columns = ["name", "null_rate", "is_clean"]
rows = [
    {"name": "revenue", "null_rate": 0.02, "is_clean": True},
    {"name": "country", "null_rate": 0.00, "is_clean": True},
]

# Write
with open("quality_report.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(rows)

# Read back
with open("quality_report.csv", "r", newline="") as f:
    for row in csv.DictReader(f):
        print(row["name"], row["null_rate"], row["is_clean"])
        
# output:
'''
revenue 0.02 True
country 0.0 True '''

# The types are strings now ("0.02", "True"), but you printed them and Python's 
# print doesn't care. If you needed numeric operations, you'd convert.

<h6>
<b><u> Part 7: When Things Get Tricky</u></b>

<b> Values containing commas</b>

If a value contains a comma, CSV wraps it in quotes:

text
<i>
name,description

Widget,"A small, handy tool"</i>

DictReader handles this automatically — it reads "A small, 
handy tool" as one field.

<b> Values containing quotes </b>

CSV escapes quotes by doubling them:

text
<i>
name,quote

Bob,"He said ""hello"""</i>

DictReader reads it as He said "hello".

<b> Missing columns in a row</b>
If a row has fewer fields than the header, DictReader fills missing values with None. If a row has more, the extra fields land under a None key (a list). Rare in well-formed files.

<b> Empty lines</b>
Blank rows are usually skipped by DictReader. If you need to detect them, use csv.reader and check for empty lists.

 <b>Different delimiters</b>
Some CSVs use tabs or semicolons. csv handles them:

** python
<i>
csv.DictReader(f, delimiter="\t")</i>

<b><u> Part 8: Comparing csv to json</u></b>

Both are text formats. Both have module functions. Both are used constantly.

Both are text formats, both have module functions, and both are used constantly.

| Aspect           | JSON                          | CSV                         |
|------------------|-------------------------------|-----------------------------|
| Structure        | Nested (dicts, lists)          | Flat (rows of columns)      |
| Types            | Preserved (int, float, bool, null) | All strings             |
| Header           | Implicit in keys               | Explicit first row           |
| Human-readable   | Yes                            | Yes                         |
| Best for         | Structured data, API payloads  | Tabular data, spreadsheets  |Rule of thumb:

<b> JSON</b> — sending structured data between systems (APIs, configs).

<b> CSV </b>— tabular data (rows of records), spreadsheets, exports.

<b><u> Part 9: Common Pitfalls</u></b>

| Pitfall                                  | What goes wrong |
|------------------------------------------|-----------------|
| Forgetting `newline=""`                  | Blank rows appear between every row on Windows. |
| Treating values as numbers               | `TypeError` — CSV values are strings. |
| Forgetting `writer.writeheader()`        | The file has no header; `DictReader` may fail later. |
| Forgetting `fieldnames=`                 | `DictWriter` raises an error. |
| Using `writerow` when you meant `writerows` | Only one row is written, or an error occurs. |
| Using `reader` when you meant `DictReader` | You get lists instead of dictionaries. |
| Not handling missing values              | `None` may show up unexpectedly. |